# Simulation A, p=16 — backend MLE

Job so sánh tốc độ backend: 20 replicate, chạy cả `tau` và `submodel` bằng backend Python MLE cũ. Kết quả dùng để so với notebook IPMS cùng cấu hình, không dùng thay kết quả parity với R.

In [ ]:
import importlib.util, subprocess, sys
required = {"rdata": "rdata>=0.11", "networkx": "networkx>=3.0", "joblib": "joblib>=1.3"}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies: OK")

In [ ]:
from pathlib import Path
import shutil, zipfile
import pandas as pd
from IPython.display import Image, display

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/backwardCGM-PD")
RESULTS = Path("/kaggle/working/simulation-mle-A-p16-results")
RESULTS.mkdir(parents=True, exist_ok=True)

checkpoint_archives = list(INPUT_ROOT.rglob("simulation-mle-A-p16-results.zip"))
checkpoint_files = list(INPUT_ROOT.rglob("simulation-mle-A-p16.json"))
if checkpoint_archives:
    with zipfile.ZipFile(checkpoint_archives[0]) as archive:
        archive.extractall(RESULTS)
    print("Restored MLE checkpoint:", checkpoint_archives[0])
elif checkpoint_files:
    shutil.copytree(checkpoint_files[0].parent, RESULTS, dirs_exist_ok=True)
    print("Restored MLE checkpoint:", checkpoint_files[0])

archives = list(INPUT_ROOT.rglob("backwardCGM-PD-kaggle-dataset.zip"))
scripts = list(INPUT_ROOT.rglob("python-port/experiments/simulation.py"))
if archives:
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(WORK_ROOT)
elif scripts:
    shutil.copytree(scripts[0].parents[2], WORK_ROOT, dirs_exist_ok=True)
else:
    raise FileNotFoundError("Hãy Add Input dataset backwardCGM-PD")
PORT_ROOT = WORK_ROOT / "python-port"
if not (PORT_ROOT / "experiments/simulation.py").exists():
    raise FileNotFoundError("Không tìm thấy experiments/simulation.py")
print("Dataset: OK\nOutput:", RESULTS)

In [ ]:
SCENARIO = "A"
P = "16"
REPLICATES = 20
METHOD = "both"
BACKEND = "mle"
PARALLEL = 1
print({"scenario": SCENARIO, "p": P, "replicates": REPLICATES, "method": METHOD, "backend": BACKEND, "parallel": PARALLEL})

In [ ]:
output = RESULTS / "simulation-mle-A-p16.json"
command = [
    sys.executable, "-u", str(PORT_ROOT / "experiments/simulation.py"),
    "--scenario", SCENARIO, "--p", P,
    "--replicates", str(REPLICATES), "--method", METHOD,
    "--source", "saved", "--alpha", "0.05", "--itmax", "500",
    "--rcon-backend", BACKEND, "--parallel", str(PARALLEL),
    "--output", str(output), "--resume",
]
print("Running:", " ".join(command))
assert BACKEND == "mle" and PARALLEL == 1
subprocess.run(command, cwd=PORT_ROOT, check=True)

In [ ]:
summary = output.with_name(f"{output.stem}-summary.csv")
if summary.exists():
    table = pd.read_csv(summary)
    table.insert(0, "backend", BACKEND)
    display(table)
figure = output.with_name(f"{output.stem}-scenario-{SCENARIO}.png")
if figure.exists():
    display(Image(filename=str(figure)))
archive = shutil.make_archive("/kaggle/working/simulation-mle-A-p16-results", "zip", root_dir=RESULTS)
print("Download:", archive)

**Checkpoint:** lưu sau mỗi replicate và chỉ resume đúng checkpoint có `rcon_backend=mle`.  
**So sánh:** dùng `mean_runtime_seconds` theo từng method để so với output IPMS cùng Scenario và p. Không trộn recovery metrics MLE vào bảng parity chính thức với R.